# Day 2: Chunking and intelligent processing for data

Welcome to Day 2 of the 7-day AI agents crash course.

In the first part of the course, we focus on **data preparation** — getting data ready before it can be used by AI agents.

## Small and large documents

**Day 1** downloaded and parsed a GitHub repo. For some use cases (e.g. the **FAQ** database), records are small enough to put **directly** into a search engine.

It is different for **Evidently** documentation: files can be very large. Example: [descriptors.mdx](https://github.com/evidentlyai/docs/blob/main/docs/library/descriptors.mdx) in the docs repo.

You *could* index a huge page as one document, but you risk **overwhelming LLMs** at query time.

## Why we prepare large documents first

- **Token limits** — models cap input length  
- **Cost** — longer prompts cost more  
- **Performance** — quality can drop with very long contexts  
- **Relevance** — only part of a long page may match a question  

So we **split** documents into smaller pieces. For RAG-style apps (more tomorrow), this is called **chunking**.

## Today’s tasks (Day 2)

1. **Simple** character-based chunking (sliding window + overlap)  
2. **Paragraph** and **section**-based chunking  
3. **Intelligent** chunking with an LLM  

For (3) you need an **OpenAI** API key or another provider (e.g. **Groq** — swap the client library; patterns are similar).

## Load data (same pipeline as Day 1)

We use **`evidentlyai/docs`** again so numbers in the lesson (e.g. document shape at index 45) stay meaningful. Run Day 1 first if you have not installed deps.

```bash
uv add requests python-frontmatter openai tqdm
uv add --dev jupyter
```

In [ ]:
import io
import zipfile

import frontmatter
import requests


def read_repo_data(repo_owner, repo_name):
    """Download and parse .md / .mdx from GitHub (main branch)."""
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    resp = requests.get(url)
    if resp.status_code != 200:
        raise RuntimeError(f"Failed to download repository: {resp.status_code}")
    out = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    for file_info in zf.infolist():
        fn = file_info.filename
        fnl = fn.lower()
        if not (fnl.endswith(".md") or fnl.endswith(".mdx")):
            continue
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)
                data = post.to_dict()
                data["filename"] = fn
                out.append(data)
        except Exception as e:
            print(f"Error processing {fn}: {e}")
    zf.close()
    return out


evidently_docs = read_repo_data("evidentlyai", "docs")
len(evidently_docs)

## 1. Simple chunking

We continue with **`evidently_docs`** from above.

Example record at index **45** (titles/descriptions can change if the repo updates; index is what the lesson uses):

```
{'title': 'LLM regression testing',
 'description': 'How to run regression testing for LLM outputs.',
 'content': 'In this tutorial, you will learn...'}
```

The **`content`** field can be tens of thousands of characters. The simplest approach: **fixed-size** chunks, e.g. length **2000**:

- Chunk 1: `[0:2000]`  
- Chunk 2: `[2000:4000]`  
- …

**Problems:** splits mid-thought, mid-sentence, related bits land in different chunks.

**Sliding window (overlap):** e.g. size **2000**, step **1000**:

- Chunk 1: `0..2000`  
- Chunk 2: `1000..3000`  
- Chunk 3: `2000..4000`  

Overlap improves continuity and retrieval when an answer spans a boundary.

In [ ]:
doc45 = evidently_docs[45]
print(doc45.get("title"), "—", len(doc45.get("content") or ""), "chars")

In [ ]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")
    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i : i + size]
        result.append({"start": i, "chunk": chunk})
        if i + size >= n:
            break
    return result


text45 = evidently_docs[45]["content"]
chunks45 = sliding_window(text45, 2000, 1000)
len(chunks45), chunks45[0]["start"], len(chunks45[-1]["chunk"])

In [ ]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop("content")
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)

len(evidently_chunks)

`doc.copy()` + `pop('content')` keeps metadata on each chunk row without duplicating the full body key name collision.

**Rough lesson numbers:** ~21 chunks for document 45; ~**575** chunks from ~**95** documents — your counts may differ slightly as the repo changes.

**Alternatives (not fully covered here):**

- **Token-based** sliding window — better for strict LLM budgets; awkward when lots of **code** is in the file.  
- **Paragraph** / **section** splits — below.  
- **AI**-guided splits — last section.

## 2. Splitting by paragraphs and sections

### Paragraphs

```python
text = evidently_docs[45]['content']
paragraphs = re.split(r"\\n\\s*\\n", text.strip())
```

Pattern **`\n\s*\n`**: blank line between blocks. Works for prose; technical docs often have **many short** “paragraphs”. You can **combine** paragraph boundaries with a sliding window for nicer chunks (good exercise).

In [ ]:
import re

text = evidently_docs[45]["content"]
paragraphs = re.split(r"\n\s*\n", text.strip())
len(paragraphs), paragraphs[:2]

### Sections (Markdown headings)

Split on lines like `## Heading` for a chosen level.

In [ ]:
import re


def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)
    parts = pattern.split(text)
    sections = []
    for i in range(1, len(parts), 3):
        header = parts[i] + parts[i + 1]
        header = header.strip()
        content = ""
        if i + 2 < len(parts):
            content = parts[i + 2].strip()
        if content:
            section = f"{header}\n\n{content}"
        else:
            section = header
        sections.append(section)
    return sections


sections = split_markdown_by_level(text, level=2)
len(sections)

In [ ]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop("content")
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc["section"] = section
        evidently_chunks.append(section_doc)

len(evidently_chunks)

## 3. Intelligent chunking with an LLM

Useful when structure is messy, you need **semantic** coherence, or custom rules — and when **quality** beats **cost**.

For most projects, **simple + overlap** or **section** splits are enough.

**Setup**

- API key: [OpenAI API keys](https://platform.openai.com/api-keys) (or Groq, etc.).  
- Set env var (Unix): `export OPENAI_API_KEY='...'`  
- Windows PowerShell: `$env:OPENAI_API_KEY = '...'`  
- **Never commit keys.** Put `OPENAI_API_KEY=...` in **`ai_hero/.env`** — the notebook loads it via `python-dotenv`. Gitignore `.env`.

```bash
uv add openai python-dotenv
```

**Cost warning:** the loop below can call the API **once per document**. Start with **`MAX_DOCS_FOR_LLM = 1`** to validate, then increase deliberately.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

_p = Path.cwd()
for _ in range(10):
    if (_p / ".env").is_file():
        load_dotenv(_p / ".env")
        break
    if _p.parent == _p:
        break
    _p = _p.parent

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "Set OPENAI_API_KEY: create ai_hero/.env with OPENAI_API_KEY=sk-... or export the variable before Jupyter."
    )

openai_client = OpenAI()


def llm(prompt, model="gpt-4o-mini"):
    response = openai_client.responses.create(model=model, input=prompt)
    return response.output_text

In [ ]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split("---")
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [ ]:
from tqdm.auto import tqdm

MAX_DOCS_FOR_LLM = 1  # increase only when you accept API cost/latency

evidently_chunks = []
subset = evidently_docs[:MAX_DOCS_FOR_LLM]

for doc in tqdm(subset):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop("content")
    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc["section"] = section
        evidently_chunks.append(section_doc)

len(evidently_chunks)

`tqdm` shows a progress bar for long runs.

**Bonus:** similar idea for **code** repos — e.g. prompt the model to summarize each class/function in plain English, then index **code + summary** together.

## 4. How to choose a chunking approach

Start **simple** (sliding window + overlap), then add structure (sections), then LLM splits if evaluations show you need them. Day 3+ will connect this to **indexing** and later **evaluations**.

## Coming up: Day 3

Data is prepared — next we **index** it (search engine) for the agent.

## Homework

- In **`project/`**, apply chunking to the repo you chose on Day 1.  
- Try **sliding window**, **paragraph + window**, and **section** splits.  
- Inspect chunks manually — what fits your app?  
- Post about what you are building (learning in public).

## Learning in public (examples)

**LinkedIn** — Day 2: chunking for your agent; three methods; insight (start simple!); repo link; [course](https://alexeygrigorev.com/aihero/).

**X** — sliding window, section splits, optional AI chunking; repo; tomorrow: agent.

## Community

[DataTalks.Club Slack](https://datatalks.club/) — `#course-ai-hero`